# LociSimiles Contextual Latin BERT Example

Run contextual token-level retrieval on the sample Latin corpus.

This notebook prefers a local LatinBERT directory at `./models/latinbert` and falls back to the HuggingFace model `ashleygong03/bamman-burns-latin-bert`.

In [3]:
from pathlib import Path
from locisimiles.document import Document
from locisimiles.evaluator import IntertextEvaluator
from locisimiles.pipeline import LatinBertRetrievalPipeline, pretty_print

In [4]:
base = Path('.')
query_doc = Document(base / 'hieronymus_samples.csv', author='Hieronymus')
source_doc = Document(base / 'vergil_samples.csv', author='Vergil')

local_latinbert_path = base / 'models' / 'latinbert'
latinbert_model_name = 'ashleygong03/bamman-burns-latin-bert'

pipeline_kwargs = {
    'top_k': 10,
    'similarity_threshold': 0.85,
    'max_length': 256,
    'min_token_length': 2,
    'use_stopword_filter': True,
}
if local_latinbert_path.exists():
    pipeline_kwargs['model_path'] = local_latinbert_path
    print(f'Using local LatinBERT model path: {local_latinbert_path}')
else:
    pipeline_kwargs['model_name'] = latinbert_model_name
    print(f'Using HuggingFace LatinBERT model: {latinbert_model_name}')

pipeline_contextual = LatinBertRetrievalPipeline(**pipeline_kwargs)

Using HuggingFace LatinBERT model: ashleygong03/bamman-burns-latin-bert


In [5]:
results_contextual = pipeline_contextual.run(query=query_doc, source=source_doc, top_k=10)
pretty_print(results_contextual)


▶ Query segment 'hier. adv. iovin. 1.1':
  verg. ecl. 8.62            candidate=+0.952  judgment=1.000
  verg. georg. 1.197         candidate=+0.945  judgment=1.000
  verg. aen. 1.177           candidate=+0.945  judgment=1.000
  verg. aen. 10.636          candidate=+0.941  judgment=1.000
  verg. georg. 2.475         candidate=+0.939  judgment=1.000
  verg. ecl. 3.49            candidate=+0.930  judgment=1.000
  verg. ecl. 3.26            candidate=+0.929  judgment=1.000
  verg. aen. 11.508          candidate=+0.925  judgment=1.000
  verg. aen. 10.875          candidate=+0.898  judgment=1.000
  verg. aen. 4.172           candidate=+0.875  judgment=1.000

▶ Query segment 'hier. adv. iovin. 1.41':
  verg. aen. 11.508          candidate=+0.933  judgment=1.000
  verg. ecl. 8.62            candidate=+0.914  judgment=1.000
  verg. ecl. 3.26            candidate=+0.913  judgment=1.000
  verg. aen. 1.177           candidate=+0.897  judgment=1.000
  verg. aen. 10.875          candidate=+0.896  

In [6]:
evaluator = IntertextEvaluator(
    query_doc=query_doc,
    source_doc=source_doc,
    ground_truth_csv=base / 'ground_truth.csv',
    pipeline=pipeline_contextual,
    top_k=10,
    threshold=0.85,
)
print(evaluator.evaluate(average='macro', with_match_only=True))

   precision  recall        f1  accuracy   fpr  fnr   smr  tp  fp  fn  tn
0   0.106667     1.0  0.192727      0.16  0.84  0.0  0.84  10  84   0   6


In [7]:
pipeline_contextual.to_csv(base / 'results_contextual_bert.csv', results=results_contextual)
pipeline_contextual.to_json(base / 'results_contextual_bert.json', results=results_contextual)